# Task 7: ResNet-50 Style Residual Bottleneck Block with Grouped/Depthwise Convolutions

In [1]:
import torch
import torch.nn as nn
from torch.profiler import profile, ProfilerActivity
torch.manual_seed(42)


In [2]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self,in_c,out_c,stride=1):
        super().__init__()
        self.depth=nn.Conv2d(in_c,in_c,3,stride,1,groups=in_c,bias=False)
        self.point=nn.Conv2d(in_c,out_c,1,bias=False)
        self.bn=nn.BatchNorm2d(out_c)
        self.act=nn.ReLU(inplace=True)
    def forward(self,x):
        return self.act(self.bn(self.point(self.depth(x))))


In [3]:
class Bottleneck(nn.Module):
    expansion=4
    def __init__(self,in_c,planes,stride=1):
        super().__init__()
        out=planes*self.expansion
        self.conv1=nn.Sequential(nn.Conv2d(in_c,planes,1,bias=False),nn.BatchNorm2d(planes),nn.ReLU(True))
        self.conv2=DepthwiseSeparableConv(planes,planes,stride)
        self.conv3=nn.Sequential(nn.Conv2d(planes,out,1,bias=False),nn.BatchNorm2d(out))
        self.skip=nn.Identity() if stride==1 and in_c==out else nn.Sequential(nn.Conv2d(in_c,out,1,stride,bias=False),nn.BatchNorm2d(out))
        self.act=nn.ReLU(True)
    def forward(self,x):
        y=self.conv1(x); y=self.conv2(y); y=self.conv3(y)
        return self.act(y+self.skip(x))


In [4]:
def count_params(m): return sum(p.numel() for p in m.parameters())
for s in [32,64,128]:
    m=Bottleneck(s,s//2)
    x=torch.randn(2,s,56,56)
    with profile(activities=[ProfilerActivity.CPU],record_shapes=True) as prof:
        y=m(x)
    print(f"Channels:{s} Output:{tuple(y.shape)} Params:{count_params(m):,}")
    print(prof.key_averages().table(sort_by='cpu_time_total',row_limit=8))


/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


Channels:32 Output:(2, 64, 56, 56) Params:4,304
--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                            Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                    aten::conv2d         1.70%       3.726ms        80.91%     176.998ms      35.400ms             5  
               aten::convolution         3.13%       6.837ms        79.20%     173.271ms      34.654ms             5  
              aten::_convolution         0.94%       2.053ms        76.08%     166.434ms      33.287ms             5  
        aten::mkldnn_convolution        52.37%     114.563ms        52.38%     114.591ms     114.591ms             1  
               aten::thnn_conv2d         0.01%      31.791us        19.97%      43.691ms      10.923ms             4  


### Notes
- Implements a ResNet-50 bottleneck (1×1 → depthwise separable 3×3 → 1×1).
- Includes learnable projection skip connection.
- Uses grouped/depthwise convolution.
- Profiles execution with `torch.profiler`.
- Parameter counts can be compared across channel scaling factors. FLOPs can be estimated from profiler/operator statistics or extended with fvcore/ptflops if desired.
